# Notebook 19 — Building a Simple Retriever
Goal: Build a reusable retrieval pipeline for longer documents.

This notebook extends the RAG basics by introducing:
- longer source documents
- chunking
- a retrieval function
- inspection of retrieval quality

## Section 1 — Install and Import Libraries

In [ ]:
# If needed:
# pip install sentence-transformers scikit-learn pandas numpy

import re
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## Section 2 — Example Longer Documents
These toy documents simulate slightly longer scientific notes.

In [ ]:
documents = [
    """KRAS is a commonly mutated oncogene in colorectal cancer. KRAS mutations can drive abnormal signaling through the MAPK pathway. These alterations are often associated with resistance to certain targeted therapies.""",

    """EGFR is an important receptor tyrosine kinase in cancer biology. EGFR inhibitors are used in several tumor types, especially lung cancer. Clinical response can depend on mutation status and pathway context.""",

    """Organoid models are increasingly used in translational cancer research. These systems can capture patient-specific drug response patterns and may help bridge experimental assays and clinical outcomes.""",

    """TP53 is a tumor suppressor gene involved in genome stability, apoptosis, and cell-cycle regulation. TP53 mutations are among the most common events across human cancers."""
]

docs_df = pd.DataFrame({"document": documents})
docs_df

## Section 3 — Why Chunking Matters
Long documents are usually split into smaller text chunks before embedding.
Retrieval then happens at the chunk level, not usually at the whole-document level.

## Section 4 — Define a Simple Chunking Function

In [ ]:
def chunk_text(text, max_sentences=2):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = " ".join(sentences[i:i+max_sentences]).strip()
        if chunk:
            chunks.append(chunk)
    return chunks

example_chunks = chunk_text(documents[0], max_sentences=2)
example_chunks

## Section 5 — Chunk All Documents

In [ ]:
all_chunks = []

for doc_id, doc in enumerate(documents):
    chunks = chunk_text(doc, max_sentences=2)
    for chunk_id, chunk in enumerate(chunks):
        all_chunks.append({
            "doc_id": doc_id,
            "chunk_id": chunk_id,
            "chunk_text": chunk
        })

chunks_df = pd.DataFrame(all_chunks)
chunks_df

## Section 6 — Create Embeddings for Chunks

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = model.encode(chunks_df["chunk_text"].tolist())

len(chunk_embeddings), chunk_embeddings[0].shape

## Section 7 — Build a Retrieval Function

In [ ]:
def retrieve(query, chunks_df, chunk_embeddings, model, top_k=3):
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]

    ranked_idx = np.argsort(similarities)[::-1][:top_k]

    results = chunks_df.iloc[ranked_idx].copy()
    results["score"] = similarities[ranked_idx]
    return results

query = "Which gene is commonly mutated in colorectal cancer?"
retrieve(query, chunks_df, chunk_embeddings, model, top_k=3)

## Section 8 — Try Different Queries

In [ ]:
query2 = "What models are used to test patient-specific drug response?"

retrieve(query2, chunks_df, chunk_embeddings, model, top_k=3)

In [ ]:
query3 = "Which pathway is associated with KRAS signaling?"

retrieve(query3, chunks_df, chunk_embeddings, model, top_k=3)

## Section 9 — Inspect Retrieval Quality
Read the top retrieved chunks and decide:
- Did retrieval find the right evidence?
- Was the best chunk ranked first?
- Were any irrelevant chunks retrieved?

In [ ]:
results = retrieve(query, chunks_df, chunk_embeddings, model, top_k=3)

for _, row in results.iterrows():
    print(f"Score: {row['score']:.3f}")
    print(row["chunk_text"])
    print("-" * 60)

## Section 10 — Retrieval Parameters
You can experiment with:
- chunk size
- top_k
- embedding model
- query wording

All of these can affect retrieval quality.

## Section 11 — Exercises

1. Change the chunking function to use 1 sentence per chunk.
2. Compare retrieval results for 1-sentence vs 2-sentence chunks.
3. Increase `top_k` from 3 to 5 and inspect the extra chunks.
4. Reword a query and see how the rankings change.
5. Add a new scientific document and rebuild the retriever.

## Skills Gained
- chunking longer documents
- building reusable retrieval functions
- retrieving chunks with embedding similarity
- inspecting ranking quality
- understanding how retrieval design affects RAG performance